# PyTorch 2.0 torch.compile & TorchInductor Deep Dive

## 1. The Memory Bandwidth Wall & Operator Fusion
Modern deep learning hardware (NVIDIA A100/H100) possesses immense compute throughput (300+ TFLOPs), but High Bandwidth Memory (HBM) bandwidth is limited (2-3 TB/s).

For **point-wise operations** (LayerNorm, GeLU, Bias Add, Residual Sum), the GPU compute cores spend most of their time waiting for data to travel across the memory bus from HBM:

$$\text{Eager Execution: } x \xrightarrow{\text{Load}} \text{BiasAdd} \xrightarrow{\text{Store}} \text{HBM} \xrightarrow{\text{Load}} \text{GeLU} \xrightarrow{\text{Store}} \text{HBM} \xrightarrow{\text{Load}} \text{LayerNorm} \xrightarrow{\text{Store}} \text{HBM}$$

Three separate kernel launches and **six full memory roundtrips** to off-chip VRAM!

### The `torch.compile` Pipeline
PyTorch 2.0 solves this via a modern compiler stack:
1. **TorchDynamo**: Uses PEP 523 Python frame evaluation to capture Python bytecode into a clean FX computation graph without breaking on dynamic Python logic.
2. **AOTAutograd**: Automatically generates joint forward and backward graphs with explicit functional primitives.
3. **TorchInductor**: Lowers the FX graph into optimized C++ (for CPU) or **Triton kernels (for GPU)**, fusing adjacent point-wise operators into a single unified kernel loop:

$$\text{Fused Kernel: } x \xrightarrow{\text{Single Load to SRAM}} [\text{BiasAdd} \to \text{GeLU} \to \text{LayerNorm}] \xrightarrow{\text{Single Store}} \text{HBM}$$

In [ ]:
import numpy as np

def simulate_eager_operations(x, bias):
    """
    Simulates eager execution with separate kernel launches and HBM roundtrips.
    """
    # Kernel 1: Bias Add (Read x, bias; Write out1 to HBM)
    out1 = x + bias
    
    # Kernel 2: GeLU Activation (Read out1; Write out2 to HBM)
    # Approx: 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
    out2 = 0.5 * out1 * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (out1 + 0.044715 * (out1 ** 3))))
    
    # Kernel 3: Scale (Read out2; Write out3 to HBM)
    out3 = out2 * 1.5
    
    # Total memory traffic: 3 Loads + 3 Stores = 6 transfers of data size
    memory_traffic_bytes = 6 * x.nbytes
    return out3, memory_traffic_bytes

def simulate_fused_inductor_kernel(x, bias):
    """
    Simulates TorchInductor fused Triton kernel.
    Loads x and bias into GPU registers/SRAM ONCE, executes all math, stores result ONCE.
    """
    # Single loop fusion in SRAM
    out_fused = np.empty_like(x)
    for i in range(len(x)):
        val = x[i] + bias[i]
        gelu_val = 0.5 * val * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (val + 0.044715 * (val ** 3))))
        out_fused[i] = gelu_val * 1.5
        
    # Total memory traffic: 1 Load (x, bias) + 1 Store (out) = 2 transfers of data size
    memory_traffic_bytes = 2 * x.nbytes
    return out_fused, memory_traffic_bytes

# Verification
N = 100000
x_arr = np.random.randn(N).astype(np.float32)
b_arr = np.random.randn(N).astype(np.float32)

res_eager, bytes_eager = simulate_eager_operations(x_arr, b_arr)
res_fused, bytes_fused = simulate_fused_inductor_kernel(x_arr, b_arr)

np.testing.assert_allclose(res_eager, res_fused, rtol=1e-5)
traffic_reduction = (1.0 - (bytes_fused / bytes_eager)) * 100

print(f"Array Size: {N} elements ({x_arr.nbytes / 1024:.1f} KB)")
print(f"Eager Execution HBM Traffic : {bytes_eager / 1024:.1f} KB (3 Kernel launches)")
print(f"Fused Inductor HBM Traffic  : {bytes_fused / 1024:.1f} KB (1 Unified Triton kernel)")
print(f"HBM Memory Bus Traffic Cut  : {traffic_reduction:.1f}% reduction!")

## 2. Graph Breaks: Diagnosis and Mitigation
A **Graph Break** occurs when TorchDynamo encounters Python constructs it cannot convert into a static computation graph (e.g., calling unsupported third-party C-extensions, logging to terminal, or inspecting raw tensor data with `item()`).
When a graph break occurs, execution falls back to the slow Python interpreter and restarts compilation for the remainder of the model.

Diagnose using:
```bash
TORCH_LOGS="+dynamo" python train.py
# Or programmatically:
explanation = torch._dynamo.explain(model, *inputs)
print(explanation.graph_break_count)
```

## 3. High-Yield Interview Questions & Best Practices

### Q1: What is the difference between `torch.jit.trace` (TorchScript) and `torch.compile`?
**Answer:** 
- **TorchScript (`torch.jit.trace`)**: Runs model with dummy inputs and records the exact linear sequence of PyTorch operations. It completely fails on Python control flow (if-statements, dynamic loops), producing silent bugs if control flow depends on inputs.
- **`torch.compile` (TorchDynamo)**: Intercepts Python frame evaluation at runtime (PEP 523). It analyzes Python bytecodes and guards assumptions. If dynamic control flow occurs, it automatically captures multiple sub-graphs guarded by checks, generating fast fused Triton code without changing user Python code.

### Q2: What does the `mode="reduce-overhead"` setting in `torch.compile` do?
**Answer:** `mode="reduce-overhead"` enables automatic **CUDA Graphs integration**. In addition to fusing operators into Triton kernels, it captures the entire sequence of compiled kernels into a static CUDA Graph, completely eliminating CPU kernel launch overhead during decoding.